# Phase 4 - Transformation, Data Quality & Analytical Dataset

Thin orchestration / narrative over `nhs_rtt.transform`. All reusable logic
lives in `src/`; this notebook only runs it and shows the evidence.

The transformation contract, the derived-field dictionary, the analytical
grains, the data-quality policy, the publication lifecycle and the determinism
contract are documented in `docs/phase4_transformation.md` (decisions `D-040`,
`D-041`; independent audit `docs/phase4_codex_audit.md`). Frozen Phase 1-3
contracts (`verify_publication`, the 105-band schema, the candidate key,
`part_2a_subset_conformance`, `mapping_diagnostics`) are **reused, not
redefined**.

Phase 4 consumes a **private immutable snapshot** of the verified Phase 3
publication; it never re-reads `data/raw/*.csv` and never mutates
`data/interim/`. Outputs go to the git-ignored `data/processed/` and are
committed as one coherent generation (`phase4_generation.json`).

In [1]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

from nhs_rtt import semantics as S
from nhs_rtt import transform as T
from nhs_rtt.crossmonth import verify_publication

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = REPO / "data" / "interim"
PROCESSED = REPO / "data" / "processed"
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
print(REPO)

C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times


## 1. Input gate - a verified *immutable snapshot* of the Phase 3 publication

`load_verified_publication` copies the publication trio into a private temp
dir, runs `crossmonth.verify_publication` against **that snapshot**, derives
the consumed digest from the same bytes, and parses only the snapshot - so the
frame, its digest and the generation identity always refer to the same
verified content even under a concurrent replace / change-and-restore
(P4-A01).

In [2]:
print("verify_publication:", verify_publication(INTERIM))
vin = T.load_verified_publication(INTERIM)
marker = json.loads((INTERIM / "rtt_combined.generation.json").read_text())
print("input rows          :", f"{vin.combined_rows:,}")
print("input columns       :", vin.df.shape[1], "(121 source + 4 provenance)")
print("phase3 generation_id:", vin.generation_id)
print("consumed sha256     :", vin.parquet_sha256)
print("digest == marker    :", vin.parquet_sha256 == marker["parquet_sha256"])
print("generation == marker:", vin.generation_id == marker["generation_id"])

verify_publication: {'valid': True, 'generation_id': 'f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d', 'combined_rows': 541363}


input rows          : 541,363
input columns       : 125 (121 source + 4 provenance)
phase3 generation_id: f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d
consumed sha256     : e5e1a0a40abf447c581895837126fe85eccd8dacc56a75e80c730c5bed2d188f
digest == marker    : True
generation == marker: True


## 2. End-to-end run - stage, validate, manifest, atomic commit

`run_phase4` = verified snapshot -> build analytical-wide + wait-band metadata
-> **stage** every output -> **mandatory** `validate_persisted_long` (streamed
from disk) -> write `phase4_generation.json` last -> atomic commit ->
self-verify. Repeated runs against the same verified generation produce
byte-identical wide / long / metadata Parquet and a stable
`phase4_generation_id`.

In [3]:
res = T.run_phase4(interim_dir=INTERIM, out_dir=PROCESSED,
                   block_rows=40_000, write_long=True, reconcile_long=False)
print(res.render_text())

=== Phase 4 — transformation & analytical dataset ===
input verified   : True  (phase3 generation f67b7342a30c4642…)
consumed sha256  : e5e1a0a40abf447c…
input rows       : 541,363
analytical-wide  : 541,363 rows × 138 cols  key unique=True
waiting-band-long: 56,843,115 rows (= wide × 105); 19.1 MB; 60.871s; peak 542.3 MiB
persisted-long   : validated 56,843,115 rows in 217 batches, 0 mismatches (44.464s)
band metadata    : 105 rows
phase4 generation: c15eecd959e2598e…  publication valid=True
  output analytical_wide: C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\rtt_analytical_wide.parquet
  output wait_band_metadata: C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\wait_band_metadata.parquet
  output waiting_band_long: C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\rtt_waiting_band_long.parquet
  output report_json: C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\pr

## 3. Coherent-publication verification (P4-A03)

`verify_phase4_publication` is the consumer gate: the manifest binds the
consumed Phase 3 identity and every output's SHA-256 / schema / rows; a
swapped old output, a tampered Parquet or a mixed set all fail.

In [4]:
print(json.dumps(T.verify_phase4_publication(PROCESSED), indent=2))
man = json.loads((PROCESSED / T.PHASE4_MANIFEST).read_text())
print("\nphase4_generation_id :", man["phase4_generation_id"])
print("phase3_input         :", man["phase3_input"])
print("long == wide * 105   :", man["long_equals_wide_times_bands"])
print("bound outputs        :", {k: v["sha256"][:12] + "..." for k, v in man["outputs"].items()})

{
  "valid": true,
  "phase4_generation_id": "c15eecd959e2598e918ffb81f524373ce3de681071cd9f6a2a4f130c77267c86",
  "phase3_generation_id": "f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d",
  "analytical_wide_rows": 541363,
  "waiting_band_long_rows": 56843115
}

phase4_generation_id : c15eecd959e2598e918ffb81f524373ce3de681071cd9f6a2a4f130c77267c86
phase3_input         : {'generation_id': 'f67b7342a30c4642a78864243142d910b84295dc7876ffb01c231cb8dc52627d', 'consumed_parquet_sha256': 'e5e1a0a40abf447c581895837126fe85eccd8dacc56a75e80c730c5bed2d188f', 'combined_rows': 541363}
long == wide * 105   : True
bound outputs        : {'analytical_wide': '474da0fe3089...', 'wait_band_metadata': '0d61346371a1...', 'waiting_band_long': '8aafa27d1a81...', 'report_json': '330376c725bc...', 'report_md': '07b5b7c93f04...'}


## 4. Analytical-wide - grain, row conservation, derived fields

One accepted Phase 3 source row -> exactly one analytical-wide row
(`541,363 -> 541,363`). The 125 source+provenance columns are carried through
verbatim (no `fillna(0)`); 13 derived columns are appended.
`reporting_period_start_date` resolution is enforced to `datetime64[us]`.

In [5]:
wide = pd.read_parquet(PROCESSED / T.ANALYTICAL_WIDE_PARQUET)
krep = S.candidate_key_report(wide, S.CANDIDATE_KEY)
print("analytical-wide :", f"{len(wide):,} rows x {wide.shape[1]} cols")
print("row conservation:", len(wide) == vin.combined_rows)
print("candidate key   : unique=%s usable=%s" % (krep["is_unique"], krep["usable"]))
print("date dtype      :", wide["reporting_period_start_date"].dtype, "==", T.REPORTING_DATE_DTYPE)
print("\nderived time cols   :", T.DERIVED_TIME_COLS)
print("derived class cols  :", T.DERIVED_CLASSIFICATION_COLS)
print("row-level dq flags  :", T.DQ_FLAG_COLS)
cols = ["reporting_month", "RTT Part Type", "Treatment Function Code",
        "Commissioner Org Code", *T.DERIVED_TIME_COLS,
        "is_treatment_function_total", "is_nonc_commissioner",
        "rtt_part_event_basis", "rtt_part_carries_bands",
        "rtt_part_is_month_end_snapshot"]
wide[cols].drop_duplicates(
    subset=["reporting_month", "RTT Part Type", "Commissioner Org Code",
            "is_treatment_function_total"]).head(12)

analytical-wide : 541,363 rows x 138 cols
row conservation: True
candidate key   : unique=True usable=True
date dtype      : datetime64[us] == datetime64[us]

derived time cols   : ['reporting_year', 'reporting_month_num', 'reporting_month_name', 'reporting_period_start_date']
derived class cols  : ['is_treatment_function_total', 'is_nonc_commissioner', 'rtt_part_event_basis', 'rtt_part_carries_bands', 'rtt_part_is_month_end_snapshot']
row-level dq flags  : ['dq_all_bands_missing', 'dq_total_missing', 'dq_part2a_gt_part2', 'dq_part2a_no_matching_part2']


,reporting_month,RTT Part Type,Treatment Function Code,Commissioner Org Code,reporting_year,reporting_month_num,reporting_month_name,reporting_period_start_date,is_treatment_function_total,is_nonc_commissioner,rtt_part_event_basis,rtt_part_carries_bands,rtt_part_is_month_end_snapshot
0,2026-04,Part_1A,C_130,06Q,2026,4,April,2026-04-01,False,False,completed_admitted_in_month,True,False
1,2026-04,Part_1A,C_999,06Q,2026,4,April,2026-04-01,True,False,completed_admitted_in_month,True,False
2,2026-04,Part_1B,C_130,06Q,2026,4,April,2026-04-01,False,False,completed_non_admitted_in_month,True,False
3,2026-04,Part_1B,C_999,06Q,2026,4,April,2026-04-01,True,False,completed_non_admitted_in_month,True,False
4,2026-04,Part_2,C_130,06Q,2026,4,April,2026-04-01,False,False,incomplete_at_month_end,True,True
5,2026-04,Part_2,C_999,06Q,2026,4,April,2026-04-01,True,False,incomplete_at_month_end,True,True
6,2026-04,Part_2A,C_130,06Q,2026,4,April,2026-04-01,False,False,incomplete_with_dta_at_month_end,True,True
7,2026-04,Part_2A,C_999,06Q,2026,4,April,2026-04-01,True,False,incomplete_with_dta_at_month_end,True,True
8,2026-04,Part_3,C_130,06Q,2026,4,April,2026-04-01,False,False,new_clock_starts_in_month,False,False
9,2026-04,Part_3,C_999,06Q,2026,4,April,2026-04-01,True,False,new_clock_starts_in_month,False,False


## 5. Missingness - blank != zero (no blanket `fillna(0)`)

In [6]:
print(json.dumps(res.report["missingness"], indent=2))

{
  "no_blanket_fillna_zero": true,
  "rows_all_bands_missing": 109474,
  "rows_total_missing": 391233,
  "rows_total_all_missing": 0,
  "long_pathway_count_source_na": 17364420,
  "long_pathway_count_explicit_zero": 35174239,
  "wide_reporting_period_start_date_na": 0,
  "long_wait_band_upper_weeks_na_per_parent": 1
}


## 6. Data-quality condition flags and P2-U7 preservation

Every `dq_` column is a **factual condition flag**. `dq_all_bands_missing` /
`dq_total_missing` are *structurally expected* for the parts that never
collect those fields; `dq_part2a_*` are *anomalous* source conditions. The two
`dq_part2a_*` flags reproduce the frozen `part_2a_subset_conformance` per-month
counts exactly. `dq_part2a_no_matching_part2` = *no matching Part_2 row with a
usable (non-null) Total All comparator*. Values are preserved, never capped.

In [7]:
row_level = res.report["row_level_quality_conditions"]
for flag in T.DQ_FLAG_COLS:
    print(f"  {flag:32s} {row_level[flag]['total']:>10,}")
print("\ndq_part2a_no_matching_part2 meaning:")
print(" ", res.report["semantic_diagnostics"]["dq_part2a_no_matching_part2_meaning"])
print("\nPart_2A subset conformance by month (frozen check):")
print(json.dumps(res.report["semantic_diagnostics"]["part_2a_subset_by_month"], indent=2))
print("\nrow-flag cross-check vs frozen conformance:")
print(json.dumps(res.report["semantic_diagnostics"]["p2u7_row_flag_cross_check"], indent=2))
wide[wide["dq_part2a_gt_part2"].fillna(False)][
    ["reporting_month", "Provider Org Code", "Commissioner Org Code",
     "Treatment Function Code", "Total All"]]

  dq_all_bands_missing                109,474
  dq_total_missing                    391,233
  dq_part2a_gt_part2                        3
  dq_part2a_no_matching_part2               8

dq_part2a_no_matching_part2 meaning:
  no matching Part_2 row with a usable (non-null) Total All comparator — covers both 'no Part_2 row at the key' and 'Part_2 row present but Total All is <NA>'; matches the frozen part_2a_subset_conformance

Part_2A subset conformance by month (frozen check):
{
  "2026-04": {
    "n_part_2a_groups": 30541,
    "n_without_matching_part_2": 5,
    "n_conformant": 30536,
    "n_violations": 0,
    "violation_keys": []
  },
  "2026-05": {
    "n_part_2a_groups": 30340,
    "n_without_matching_part_2": 3,
    "n_conformant": 30336,
    "n_violations": 1,
    "violation_keys": [
      "NT230/05V/C_100(2A=2>2=1)"
    ]
  },
  "2026-06": {
    "n_part_2a_groups": 30548,
    "n_without_matching_part_2": 0,
    "n_conformant": 30546,
    "n_violations": 2,
    "violation_keys": 

,reporting_month,Provider Org Code,Commissioner Org Code,Treatment Function Code,Total All
238550,2026-05,NT230,05V,C_100,2
392823,2026-06,RTG,84H,C_502,2
392824,2026-06,RTG,84H,C_999,2


## 7. C_999 / NONC coverage and mapping diagnostics

Legitimate semantic classifications, not DQ conditions. Mapping churn is
diagnostic information (codes identify; names label).

In [8]:
sd = res.report["semantic_diagnostics"]
print("C_999 coverage:", json.dumps(sd["c999_coverage_by_month"], indent=2))
print("NONC coverage :", json.dumps(sd["nonc_coverage_by_month"], indent=2))
print("\nmapping diagnostics summary:")
print(json.dumps(res.report["dataset_level_diagnostics"]["mapping"], indent=2, default=str))

C_999 coverage: {
  "2026-04": {
    "rows": 40102,
    "share_pct": 22.183
  },
  "2026-05": {
    "rows": 39677,
    "share_pct": 22.269
  },
  "2026-06": {
    "rows": 40636,
    "share_pct": 22.277
  }
}
NONC coverage : {
  "2026-04": {
    "rows": 2955,
    "share_pct": 1.635
  },
  "2026-05": {
    "rows": 2904,
    "share_pct": 1.63
  },
  "2026-06": {
    "rows": 2993,
    "share_pct": 1.641
  }
}

mapping diagnostics summary:
{
  "code_name_changes": 0,
  "name_code_collisions": [
    {
      "dimension": "provider",
      "name": "DUCHY HOSPITAL",
      "n_distinct_codes": 2,
      "codes": "NT447,NVC04",
      "scope": "within_month"
    }
  ],
  "membership_changes": [
    {
      "dimension": "provider",
      "from_month": "2026-04",
      "to_month": "2026-05",
      "n_appeared": 4,
      "n_disappeared": 1
    },
    {
      "dimension": "provider",
      "from_month": "2026-05",
      "to_month": "2026-06",
      "n_appeared": 5,
      "n_disappeared": 0
    },
    {


## 8. Wait-band metadata (derived from the frozen band contract)

105 rows, derived from `semantics.expected_week_band_names()` /
`parse_week_band()` - not a second hand-maintained list. Day ranges are
**labelled DERIVED** (P2-U6). The `>104` band is open-ended (`upper` is `<NA>`).

In [9]:
meta = pd.read_parquet(PROCESSED / T.WAIT_BAND_METADATA_PARQUET)
print(meta.shape)
pd.concat([meta.head(3), meta.tail(2)])

(105, 7)


,wait_band_order,wait_band_label,wait_band_lower_weeks,wait_band_upper_weeks,wait_band_lower_days,wait_band_upper_days,is_open_ended
0,1,Gt 00 To 01 Weeks SUM 1,0,1,0,7,False
1,2,Gt 01 To 02 Weeks SUM 1,1,2,8,14,False
2,3,Gt 02 To 03 Weeks SUM 1,2,3,15,21,False
103,104,Gt 103 To 104 Weeks SUM 1,103,104,722,728,False
104,105,Gt 104 Weeks SUM 1,104,<NA>,729,<NA>,True


## 9. Waiting-band-long - dense grain & persisted-output validation

`len(long) == len(wide) * 105 == 56,843,115`. `run_phase4` already ran the
**authoritative** `validate_persisted_long` (bounded-memory, streamed
positionally from the on-disk Parquet: schema, row count, band order/label/
bounds, all nine identity fields, `pathway_count` value + zero/NA state).
Below: that result, plus the read-back cardinality and a heavier **in-memory**
`reconcile_long_against_wide` cross-check on the smallest month's slice.

In [10]:
print("authoritative persisted-long validation (from run_phase4):")
print(json.dumps(res.persisted_validation, indent=2))

pf = pq.ParquetFile(str(PROCESSED / T.WAITING_BAND_LONG_PARQUET))
expected = len(wide) * T.N_WAIT_BANDS
print("\nlong rows (read-back):", f"{pf.metadata.num_rows:,}",
      "| expected:", f"{expected:,}",
      "| match:", pf.metadata.num_rows == expected)
print("long columns:", T.LONG_COLUMNS)

month = min(res.report["input_integrity"]["source_months"],
            key=lambda m: int((wide["reporting_month"] == m).sum()))
w_m = wide[wide["reporting_month"] == month]
long_m = pd.read_parquet(PROCESSED / T.WAITING_BAND_LONG_PARQUET,
                         filters=[("reporting_month", "=", month)])
rec = T.reconcile_long_against_wide(w_m, long_m)
print(f"\nin-memory reconcile for {month} (secondary diagnostic):")
print(json.dumps(rec, indent=2))

authoritative persisted-long validation (from run_phase4):
{
  "path": "C:\\Users\\abdul\\OneDrive\\Desktop\\P_Projects\\NHS-RTT-Waiting-Times\\data\\processed\\.staging-5636-1788977925881762500\\rtt_waiting_band_long.parquet",
  "rows": 56843115,
  "expected_rows": 56843115,
  "parents": 541363,
  "bands_per_parent": 105,
  "batches": 217,
  "schema_ok": true,
  "numeric_cells": 39478695,
  "explicit_zero_cells": 35174239,
  "positive_cells": 4304456,
  "source_na_cells": 17364420,
  "value_mismatches": 0,
  "na_state_mismatches": 0,
  "identity_mismatches": 0,
  "seconds": 44.464
}

long rows (read-back): 56,843,115 | expected: 56,843,115 | match: True
long columns: ['reporting_month', 'Period', 'Provider Org Code', 'Commissioner Org Code', 'RTT Part Type', 'Treatment Function Code', 'source_file', 'source_sha256', 'source_row_index', 'wait_band_order', 'wait_band_label', 'wait_band_lower_weeks', 'wait_band_upper_weeks', 'pathway_count']



in-memory reconcile for 2026-05 (secondary diagnostic):
{
  "cells": 18707955,
  "both_missing": 5705367,
  "both_numeric": 13002588,
  "explicit_zero": 11580068,
  "duplicate_keys": 0,
  "identity_mismatches": 0,
  "value_mismatches": 0,
  "na_state_mismatches": 0
}


## 10. Dense-long benchmark & determinism contract

Implemented and measured, not guessed. Dense long is **practical** (~19 MB
Parquet, ~0.5 GiB peak, tens of seconds) and **retained**. Determinism is
**logical** (identical schema/rows/values/NA/order/lineage across any valid
`block_rows`) and, separately, **physical byte repeatability** only under a
fixed writer + environment + `block_rows` (the long file's hash changes with
`block_rows`; wide/metadata do not).

In [11]:
print(json.dumps(res.report["dense_long_benchmark"], indent=2))
print("\ndeterminism contract:")
print(json.dumps(res.report["determinism"], indent=2))

# logical determinism across block sizes on this real data (wide/meta byte-stable)
import hashlib, tempfile
def _sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
alt = Path(tempfile.mkdtemp())
r_alt = T.run_phase4(interim_dir=INTERIM, out_dir=alt, block_rows=25_000)
print("\nwide  byte-identical across block_rows:",
      _sha(PROCESSED / T.ANALYTICAL_WIDE_PARQUET) == _sha(alt / T.ANALYTICAL_WIDE_PARQUET))
print("meta  byte-identical across block_rows:",
      _sha(PROCESSED / T.WAIT_BAND_METADATA_PARQUET) == _sha(alt / T.WAIT_BAND_METADATA_PARQUET))
print("long  byte-identical across block_rows:",
      _sha(PROCESSED / T.WAITING_BAND_LONG_PARQUET) == _sha(alt / T.WAITING_BAND_LONG_PARQUET),
      "(expected False - row-group layout differs)")
print("phase4_generation_id stable          :",
      res.phase4_generation_id == r_alt.phase4_generation_id)
print("deterministic_report_view equal      :",
      T.deterministic_report_view(res.report) == T.deterministic_report_view(r_alt.report))

{
  "path": "C:\\Users\\abdul\\OneDrive\\Desktop\\P_Projects\\NHS-RTT-Waiting-Times\\data\\processed\\.staging-5636-1788977925881762500\\rtt_waiting_band_long.parquet",
  "rows": 56843115,
  "expected_rows": 56843115,
  "columns": [
    "reporting_month",
    "Period",
    "Provider Org Code",
    "Commissioner Org Code",
    "RTT Part Type",
    "Treatment Function Code",
    "source_file",
    "source_sha256",
    "source_row_index",
    "wait_band_order",
    "wait_band_label",
    "wait_band_lower_weeks",
    "wait_band_upper_weeks",
    "pathway_count"
  ],
  "parquet_bytes": 19088167,
  "block_rows": 40000,
  "n_blocks": 14,
  "build_write_seconds": 60.871,
  "peak_tracemalloc_mib": 542.3,
  "row_group_rows": 1048576,
  "read_back_rows": 56843115,
  "zero_count": 35174239,
  "na_count": 17364420,
  "cardinality_ok": true
}

determinism contract:
{
  "logical": "Across any valid block_rows (including a non-divisible final block): identical schema, rows, values, NA states, analytic


wide  byte-identical across block_rows: True
meta  byte-identical across block_rows: True
long  byte-identical across block_rows: False (expected False - row-group layout differs)
phase4_generation_id stable          : True
deterministic_report_view equal      : True


## 11. Outputs & the human-readable report

All under `data/processed/` (git-ignored, regenerable) and committed as one
coherent generation. Frozen Phase 1-3 work is untouched; `phase-4-pass` is not
created here (awaits independent closure).

In [12]:
for k, v in res.outputs.items():
    print(f"{k:20s} {v}")
print("\n" + "=" * 72)
print((PROCESSED / T.TRANSFORM_REPORT_MD).read_text(encoding="utf-8"))

analytical_wide      C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\rtt_analytical_wide.parquet
wait_band_metadata   C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\wait_band_metadata.parquet
waiting_band_long    C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\rtt_waiting_band_long.parquet
report_json          C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\phase4_transformation_report.json
report_md            C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\phase4_transformation_report.md
manifest             C:\Users\abdul\OneDrive\Desktop\P_Projects\NHS-RTT-Waiting-Times\data\processed\phase4_generation.json

# Phase 4 — transformation & data-quality summary

_Generated 2026-09-09T18:20:34+00:00. Reproducible from the verified Phase 3 publication; no substantive waiting-time analysis._

## Input integrity
- Phase 3 publicati